In [ ]:
%pip install httpx mdmodels==0.3.0 rich

In [9]:
import rich
import httpx
import mdmodels as md

In [6]:
library = md.DataModel.from_markdown("model.md")

print(library)

Library[EnumType]
  - Role
  - Molecule
  - Enzyme
  - Conditions
  - Experiment



In [8]:
# Create a new experiment with basic metadata and conditions
# This represents a complete biological assay with defined parameters
dataset = library.Experiment(
    name="Experiment 1",
    description="This is a test experiment",
    conditions=library.Conditions(
        temperature=37,  # Temperature in °C for optimal enzyme activity
        pH=7.4,  # Physiological pH level
    ),
)

# Add a molecule to the experiment as a substrate
# Molecules represent chemical compounds involved in the reaction
dataset.add_to_molecules(
    name="Molecule 1",
    initial_concentration=100,  # Starting concentration in mM or μM
    role=library.Role.SUBSTRATE,  # This molecule acts as a substrate
)

# Add an enzyme to catalyze the reaction
# Enzymes are biological catalysts that facilitate chemical reactions
dataset.add_to_enzymes(
    name="Enzyme 1",
    initial_concentration=100,  # Starting enzyme concentration in mg/mL or μM
    role=library.Role.ENZYME,  # This protein acts as the primary catalyst
)

rich.print(dataset)

Experiment(
    name='Experiment 1',
    description='This is a test experiment',
    molecules=[Molecule(name='Molecule 1', initial_concentration=100.0, role=<Role.SUBSTRATE: 'substrate'>)],
    enzymes=[Enzyme(name='Enzyme 1', initial_concentration=100.0, role=<Role.ENZYME: 'enzyme'>)],
    conditions=Conditions(temperature=37.0, pH=7.4)
)

In [ ]:
with httpx.Client() as client:
    response = client.post(
        "http://localhost:8800/experiment/",
        json=dataset.model_dump(mode="json"),
    )
    response.raise_for_status()

    rich.print(response.json())


{'id': 21, 'name': 'Experiment 1', 'description': 'This is a test experiment'}

In [ ]:
with httpx.Client() as client:
    response = client.get("http://localhost:8800/experiment/")
    datasets = [library.Experiment(**dataset) for dataset in response.json()]

rich.print(f"Found {len(datasets)} datasets")
names = [dataset.name for dataset in datasets]
rich.print("\n - " + "\n - ".join(names))

Found 21 datasets

- Hexokinase Glucose Phosphorylation - Low Substrate
 - Hexokinase Glucose Phosphorylation - Medium Substrate
 - Hexokinase Glucose Phosphorylation - High Substrate
 - Pyruvate Kinase - Low PEP
 - Pyruvate Kinase - High PEP
 - Glucose Oxidase - Low Glucose
 - Glucose Oxidase - Medium Glucose
 - Glucose Oxidase - High Glucose
 - Gluconeogenesis - Pyruvate to Glucose (Low)
 - Gluconeogenesis - Pyruvate to Glucose (High)
 - LDH - Pyruvate to Lactate (Low)
 - LDH - Pyruvate to Lactate (High)
 - ADH Ethanol Oxidation - Low Ethanol
 - ADH Ethanol Oxidation - High Ethanol
 - Hexokinase - G6P Product Inhibition
 - Glucose Isomerase - Medium Glucose
 - Glucose Isomerase - Saturating Glucose
 - ATPase Hydrolysis - Medium ATP
 - ATPase Hydrolysis - High ATP
 - Experiment 1
 - Experiment 1